# Phase 4: Custom Optimization Engine

This notebook covers:
1. **Scratch Loss Functions**: Mean Squared Error (MSE) & Log Loss / Binary Cross-Entropy (BCE)
2. **Gradient Descent Algorithms**: Batch GD, Stochastic GD (with Momentum), Mini-Batch GD
3. **Adaptive Learning Optimizers**: Adagrad, RMSprop, Adam (with bias correction)
4. **Learning Rate Schedules**: Step Decay, Exponential Decay, Cosine Annealing, Cyclical LR
5. **Trajectory Visualization**: Comparing empirical convergence rates

In [ ]:
from mlo.data.generator import generate_customer_dataset
from mlo.data.encoder import CategoricalEncoder
from mlo.data.preprocessor import DataPreprocessor
from mlo.optimization.optimizers import (
    BatchGradientDescent,
    StochasticGradientDescent,
    MiniBatchGradientDescent,
    AdagradOptimizer,
    RMSpropOptimizer,
    AdamOptimizer
)
from mlo.optimization.schedulers import CosineAnnealingScheduler
from mlo.optimization.engine import OptimizationEngine
from mlo.utils.visualization import plot_loss_curves
import numpy as np

## 1. Prepare Scaled Training Data

In [ ]:
df = generate_customer_dataset(n_samples=600, random_state=42)
encoder = CategoricalEncoder()
df_encoded = encoder.fit_transform(df)
X = df_encoded[encoder.feature_columns_].values
y = df_encoded['Churn'].values

preproc = DataPreprocessor(scaler_type="standard")
X_scaled = preproc.fit_transform(X)

## 2. Train Custom Optimization Engines

In [ ]:
epochs = 60
optimizers = {
    "Batch GD": BatchGradientDescent(learning_rate=0.1),
    "SGD (Momentum)": StochasticGradientDescent(learning_rate=0.01, momentum=0.9),
    "Mini-Batch GD": MiniBatchGradientDescent(learning_rate=0.05, batch_size=32),
    "Adagrad": AdagradOptimizer(learning_rate=0.1),
    "RMSprop": RMSpropOptimizer(learning_rate=0.01),
    "Adam": AdamOptimizer(learning_rate=0.02)
}

loss_histories = {}
for name, opt in optimizers.items():
    scheduler = CosineAnnealingScheduler(initial_lr=0.05, total_epochs=epochs) if name == "Adam" else None
    engine = OptimizationEngine(optimizer=opt, loss_type="bce", epochs=epochs, scheduler=scheduler)
    engine.fit(X_scaled, y)
    loss_histories[name] = engine.loss_history
    print(f"{name:16s} | Final Loss: {engine.loss_history[-1]:.4f}")

## 3. Compare Convergence Trajectories

In [ ]:
plot_loss_curves(loss_histories, title="Convergence Trajectories Across Custom Optimizers")